In [ ]:
#![no_std]
use soroban_sdk::{
    contract, contracterror, contractevent, contractimpl, contracttype, token, Address, Bytes,
    BytesN, Env, FromVal, IntoVal, String, Symbol, TryFromVal, Val, Vec,
};

#[cfg(test)]
extern crate std;

#[cfg(test)]
std::thread_local! {
    static TEST_TRIPPED: core::sync::atomic::AtomicBool = core::sync::atomic::AtomicBool::new(false);
    static TEST_TRIP_COUNT: core::sync::atomic::AtomicU32 = core::sync::atomic::AtomicU32::new(0);
    static TEST_RESETS_AT: core::sync::atomic::AtomicU64 = core::sync::atomic::AtomicU64::new(0);
}

// Issue #138 workaround: Using tuple-based storage keys with Symbol
// to avoid LengthExceedsMax error from large #[contracttype] enums
pub type StorageKey = (Symbol, Option<Address>, Option<u64>, Option<u32>);

/// Construct a tuple-based storage key from its components.
///
/// Uses `Symbol::new` with `Env::default()` to create the prefix symbol.
///
/// # Arguments
/// * `prefix` - A string prefix for the storage key.
/// * `addr` - An optional address component.
/// * `id` - An optional numeric ID component.
/// * `sub_id` - An optional sub-ID component.
///
/// # Returns
/// A `StorageKey` tuple suitable for use in contract storage.
pub fn make_key(
    prefix: &str,
    addr: Option<Address>,
    id: Option<u64>,
    sub_id: Option<u32>,
) -> StorageKey {
    (Symbol::new(&Env::default(), prefix), addr, id, sub_id)
}

// Legacy DataKey - split into functional groups to avoid LengthExceedsMax
#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum DataKey {
    Admin,
    Refund(u64),
    RefundCounter,
    RefundsByStatus(RefundStatus, u64),
    RefundStatusCount(RefundStatus),
    RefundStatusIndex(u64),
    MerchantRefunds(Address, u64),
    MerchantRefundQuota(Address),
    MerchantRefundCount(Address),
    CustomerRefunds(Address, u64),
    CustomerRefundCount(Address),
    // Issue: bound unbounded per-customer history growth by archiving old entries
    CustomerRefundHistoryStart(Address),
    CustomerRefundsArchive(Address, u64),
    PaymentRefunds(u64, u64),
    PaymentRefundCount(u64),
    PoolToken(u64),
    DefaultRefundPolicy,
    RefundPolicy(Address),
    // Policy versioning (#134)
    RefundPolicyVersion(Address, u32),
    RefundPolicyVersionCount(Address),
    RefundPolicyTemplate(u64),
    RefundPolicyTemplateCount,
    // Payment contract address (#143)
    PaymentContractAddress,
    BatchRefundLimit,
    RefundAnalyticsKey,
    // Rate limiting
    CustomerRefundRateLimit(Address),
    GlobalRefundRateLimit,
    // Admin override audit log
    AdminOverrideHistory(u64),
    AdminOverrideHistoryCount,
    // Payment refund caps
    PaymentRefundCap(u64),
    PaymentRefundUsage(u64),
    // Issue #370: Customer-tier-based refund caps
    CustomerTier(Address),
    CustomerTierPolicy(Address, u32),
    StrictTierPolicy(Address),
    AppealWindowSeconds,
}

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum ArbitrationKey {
    ArbitrationCase(u64),
    ArbitrationCounter,
    ArbitratorList,
    ArbitratorsVoted(u64),
    ArbitratorVote(u64, Address),
    ArbitrationFeeConfig,
    AccumulatedTreasuryFees,
    ArbitrationStakeConfig,
    ArbitrationStake(u64),
    ArbitratorReputation(Address),
    ArbitratorScoreIndex(i128, u64),
    ArbitratorScoreCount,
    ArbitrationTimeoutConfig,
    // Issue #194: Tiered arbitration
    SeniorArbitratorList,
    ArbitrationTierConfig,
    CaseEscalated(u64),
}

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum PolicyKey {
    RefundPolicyVersion(Address, u32),
    RefundPolicyVersionCount(Address),
    AutoRefundTrigger(u64),
    AutoRefundTriggerCounter,
}

// Maximum number of a customer's refund references kept in "hot" instance
// storage. Older entries are moved to persistent storage (archived) so a
// customer's history can grow indefinitely without bloating the instance
// storage footprint read/written on every contract invocation.
const CUSTOMER_HISTORY_HOT_CAP: u64 = 50;

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum SystemKey {
    PauseStateKey,
    PauseHistoryEntry(u64),
    PauseHistoryCount,
    CircuitBreakerConfigKey,
    CircuitBreakerStateKey,
    WindowStart,
    WindowRefundVolume,
    WindowPaymentVolume,
    FraudSignal(Address),
    FraudConfig,
    FlaggedAddressesIndex,
    RefundRejectedAt(u64),
    Appeal(u64),
    AppealCounter,
    AppealByRefund(u64),
    AppealByCustomer(Address, u64),
    AppealByCustomerCount(Address),
    // Notification hooks
    NotificationHook(u64),
    NotificationHookCounter,
    HooksByEvent(RefundEventType, u64),
    HooksByEventCount(RefundEventType),
    SubscriberHooks(Address, u64),
    SubscriberHookCount(Address),
    // Platform fee deduction on refund processing
    RefundFeeConfig,
    AccumulatedRefundFees,
    // Per-customer refund cooldown
    CustomerRefundCooldown(Address),
    RefundCooldownConfig,
    SchemaVersion,
}

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum EvidenceKey {
    Evidence(u64, Address),
    EvidenceIndex(u64, u64),
    EvidenceCount(u64),
}

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum VoucherKey {
    Voucher(u64),
    VoucherCounter,
    CustomerVoucher(Address, u64),
    CustomerVoucherCount(Address),
}

#[derive(Clone, Debug, PartialEq)]
#[contracttype]
pub enum TokenKey {
    SupportedToken(Address),
    TokenCount,
    TokenByIndex(u64),
}

#[derive(Clone, Debug, PartialEq, Eq)]
#[contracttype]
pub enum RefundStatus {
    Requested,
    Approved,
    Rejected,
    Processed,
    PendingAppeal,
}

// Issue #397: canonical reason codes, enforced by the type system on Refund and
// on request_refund()'s signature, so get_reason_code_analytics() never sees a
// free-form/inconsistent string for this field.
#[derive(Clone, Debug, PartialEq, Eq)]
#[contracttype]
pub enum RefundReasonCode {
    ProductDefect,
    NonDelivery,
    DuplicateCharge,
    Unauthorized,
    CustomerRequest,
    Other,
}

// Issue #138 (recurred): the flat `Error` enum grew past Soroban's 50-variant
// XDR spec limit (`VecM<ScSpecUdtErrorEnumCaseV0, 50>`), which makes the
// `#[contracterror]` macro panic with `LengthExceedsMax` at compile time.
// Split into two `#[contracterror]` enums (each <= 50 variants), wrapped by a
// single `Error` type so every existing `Result<_, Error>` signature and `?`
// call site is unaffected. Mirrors the same pattern already used for
// `Error`/`BasicError`/`EscrowError`/`ActionError` in contracts/escrow/src/lib.rs.
#[contracterror]
#[derive(Clone, Copy, Debug, PartialEq)]
pub enum CoreError {
    InvalidAmount = 1,
    RefundNotFound = 2,
    Unauthorized = 3,
    InvalidPaymentId = 4,
    InvalidStatus = 7,
    AlreadyProcessed = 8,
    RefundExceedsPayment = 9,
    TotalRefundsExceedPayment = 10,
    RefundWindowExpired = 11,
    RefundExceedsPolicy = 12,
    PolicyNotFound = 13,
    PolicyInactive = 14,
    QuorumNotReached = 15,
    NotArbitrator = 16,
    ContractPaused = 17,
    FunctionPaused = 18,
    CaseNotTimedOut = 19,
    BatchRefundTooLarge = 20,
    // Issue #138: Refund policy inheritance errors
    CircularInheritance = 21,
    MaxInheritanceDepth = 22,
    RefundNotRejected = 23,
    AppealWindowExpired = 24,
    AppealAlreadyFiled = 25,
    RefundRateLimitExceeded = 26,
    PaymentContractNotSet = 27,
    PaymentOwnershipMismatch = 28,
    CircuitBreakerTripped = 29,
    InvalidFeeConfig = 30,
    InsufficientTreasuryFees = 31,
}

#[contracterror]
#[derive(Clone, Copy, Debug, PartialEq)]
pub enum ExtError {
    ArbitratorNotFound = 34,
    InvalidScoreThreshold = 35,
    AutoRefundTriggerNotFound = 36,
    DuplicateAutoRefundTrigger = 37,
    AddressFlaggedForFraud = 38,
    FraudSignalNotFound = 40,
    // Issue #144: Notification hook errors
    HookNotFound = 41,
    MaxHooksPerEventReached = 42,
    HookNotOwnedBySubscriber = 43,
    // Issue #373: Invalid notification hook subscriber address
    // (moved from 58, which collided with SchemaAlreadyAtTarget)
    InvalidHookAddress = 51,
    // Issue #148: Customer eligibility errors
    CustomerBlockedFromRefund = 44,
    EligibilityEntryNotFound = 45,
    TemplateNotFound = 46,
    TemplateInactive = 47,
    // Issue #XXX: Payment refund cap errors
    RefundCountCapExceeded = 48,
    RefundAmountCapExceeded = 49,
    UnsupportedRefundToken = 50,
    // New specific errors
    VoucherNotFound = 52,
    VoucherExpired = 53,
    VoucherAlreadyRedeemed = 54,
    EvidenceAlreadySubmitted = 55,
    CaseAlreadyEscalated = 56,
    // Issue #370: Customer tier policy errors
    TierPolicyNotFound = 57,
    SchemaAlreadyAtTarget = 58,
}

#[derive(Clone, Copy, Debug, PartialEq)]
pub enum Error {
    Core(CoreError),
    Ext(ExtError),
}

impl Error {
    pub fn to_u32(&self) -> u32 {
        match self {
            Error::Core(e) => *e as u32,
            Error::Ext(e) => *e as u32,
        }
    }
}

impl From<Error> for soroban_sdk::Error {
    fn from(e: Error) -> Self {
        soroban_sdk::Error::from_contract_error(e.to_u32())
    }
}

impl From<&Error> for soroban_sdk::Error {
    fn from(e: &Error) -> Self {
        soroban_sdk::Error::from_contract_error(e.to_u32())
    }
}

impl TryFrom<soroban_sdk::Error> for Error {
    type Error = soroban_sdk::Error;
    fn try_from(error: soroban_sdk::Error) -> Result<Self, Self::Error> {
        if let Ok(e) = CoreError::try_from(error) {
            return Ok(Error::Core(e));
        }
        if let Ok(e) = ExtError::try_from(error) {
            return Ok(Error::Ext(e));
        }
        Err(error)
    }
}

impl TryFrom<&soroban_sdk::Error> for Error {
    type Error = soroban_sdk::Error;
    fn try_from(error: &soroban_sdk::Error) -> Result<Self, Self::Error> {
        <Self as TryFrom<soroban_sdk::Error>>::try_from(*error)
    }
}

impl FromVal<Env, Error> for Val {
    fn from_val(env: &Env, v: &Error) -> Self {
        soroban_sdk::Error::from(v).into_val(env)
    }
}

impl TryFromVal<Env, Val> for Error {
    type Error = soroban_sdk::ConversionError;
    fn try_from_val(env: &Env, val: &Val) -> Result<Self, Self::Error> {
        let error: soroban_sdk::Error =
            soroban_sdk::Error::try_from_val(env, val).map_err(|_| soroban_sdk::ConversionError)?;
        Error::try_from(error).map_err(|_| soroban_sdk::ConversionError)
    }
}